### import


In [11]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance,PayloadSchemaType,PointStruct,MatchAny,FieldCondition,Filter,Prefetch, FusionQuery

import pandas as pd
import numpy as np
import openai
import json
import tiktoken

#### Retrive all item ids from amazon items qdrant collections

In [19]:
qdrant_client = QdrantClient(url="http://localhost:6333")



In [20]:
dummy_vector=np.zeros(384).tolist()

In [21]:
payload=qdrant_client.query_points(
        collection_name="amazon-items-collection-01-hybrid-search",
        
        query=dummy_vector,
        using="all-MiniLM-L6-v2",
        limit=1000,
        with_payload=["parent_asin"],
        with_vectors=False,
)


In [22]:
payload.points

[ScoredPoint(id=382, version=3, score=0.0, payload={'parent_asin': 'B0BD5RF5J3'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=10, version=3, score=0.0, payload={'parent_asin': 'B09XBGM4GB'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=326, version=3, score=0.0, payload={'parent_asin': 'B07QN1VF48'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=43, version=3, score=0.0, payload={'parent_asin': 'B09X1WGNZX'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=196, version=3, score=0.0, payload={'parent_asin': 'B0BCKWP6QH'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=51, version=3, score=0.0, payload={'parent_asin': 'B0BJ1CVBQY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=108, version=3, score=0.0, payload={'parent_asin': 'B09TTC3H88'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=203, version=3, score=0.0, payload={'parent_asin': 'B0BMW24KRP'}, vector=None

In [23]:
len(payload.points)

1000

In [24]:
parent_asin_list = [point.payload["parent_asin"] for point in payload.points]

In [25]:
parent_asin_list

['B0BD5RF5J3',
 'B09XBGM4GB',
 'B07QN1VF48',
 'B09X1WGNZX',
 'B0BCKWP6QH',
 'B0BJ1CVBQY',
 'B09TTC3H88',
 'B0BMW24KRP',
 'B0B69PC4TP',
 'B09WRRTHND',
 'B09S6TY2T8',
 'B0BJVVSLC1',
 'B0BTW59YJJ',
 'B0BHJFB321',
 'B09M527FC6',
 'B0B6J7W6MJ',
 'B0B4F8RJCL',
 'B0BFRM3DSL',
 'B0BJFPFR2Z',
 'B09QRXY289',
 'B09YTJPYM4',
 'B0BD4XPYYM',
 'B0B6J9GBZT',
 'B0B5PXSQJH',
 'B0BVDSJLRR',
 'B0BYHFG53W',
 'B0B18XYVLS',
 'B0BT88SYXN',
 'B09PVNF18V',
 'B0C1W8D88H',
 'B0B5221X8Q',
 'B0B92FZQZB',
 'B09WPJRSRJ',
 'B09XT4VVP3',
 'B09Q12DWG2',
 'B0B15JS7FY',
 'B09PVHGSRZ',
 'B0BC2HT1BY',
 'B09Q6DK8VJ',
 'B0B3WFFR82',
 'B09TS84LC3',
 'B0B2WMXB6C',
 'B09Z82KFR1',
 'B0B3M8PDRX',
 'B09VY67S6P',
 'B09YQLN1XN',
 'B09KZRWTXS',
 'B09R34XHJ6',
 'B0B9Y5BV8G',
 'B09PRMQP5C',
 'B0BY32TVLL',
 'B0BDMZQNB8',
 'B0BV5WY5TC',
 'B09GXK8NZ7',
 'B09T8F2ZQF',
 'B09QH11JP7',
 'B09RPHV5PH',
 'B09Z6TY5F1',
 'B0BY1J3R2F',
 'B09TMWJ38M',
 'B09TJ4L1GG',
 'B0B4PJ64KC',
 'B0C55KSG91',
 'B09ZHFPB6V',
 'B0BHKWTTR1',
 'B099ZPJFZ3',
 'B0BV8FDC

#### load amazon reviews dataset


In [30]:
df_reviews=pd.read_json("../../data/CDs_and_Vinyl_2022_2023_with_category_ratings_10_sample_1000.jsonl",lines=True)

In [31]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,"If this is rock 'n' roll retirement, I'll take...","Yet again, David Crosby keeps the winning stre...",[],B0BH3R9D4X,B0BH3R9D4X,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2023-01-08 22:56:15.968,8,True
1,4,"CCR's legendary 1970 Royal Albert Hall gig, fi...",CCR's legendary 1970 Royal Albert Hall concert...,[],B0B18Z8GL1,B0B18Z8GL1,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-09-16 18:05:34.559,21,True
2,3,Songs for Beginners / Wild Tales 50 + years do...,So many 5-star reviews. I just don't get it. ...,[],B09VCS9Q5W,B09VCS9Q5W,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-06-03 20:56:32.958,4,True
3,4,A master lesson from a pair of legends,Mavis and Levon. Staples and Helm. If you're...,[],B09VLCNB1F,B09VLCNB1F,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-05-22 15:32:49.839,9,True
4,3,"Disappointing, at best","Musically and vocally, Van's still the man. B...",[],B09V4R82C4,B09V4R82C4,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-05-22 15:03:02.428,1,True


In [32]:
len(df_reviews)

8721

In [33]:
df_reviews_sample=df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]

In [34]:
len(df_reviews_sample)

8721

#### define funtions to preprocess reviews data

In [35]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"


In [ ]:
encoding=tiktoken.encoding_for_model("text-embedding-3-large")